## Amazon 变体 / 选项示例（测试用 ASIN）

| 场景 | 链接 |
|------|------|
| 无选项 | [B0BX418N68](https://www.amazon.com/dp/B0BX418N68) |
| 一个选项 | [B07Z7FSHL8](https://www.amazon.com/dp/B07Z7FSHL8) |
| 两个选项 | [B0G6CV4D4C](https://www.amazon.com/dp/B0G6CV4D4C) |
| 两个选项（其中一个为下拉） | [B0DC5NDFXJ](https://www.amazon.com/dp/B0DC5NDFXJ) |

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
from lxml import etree
from selenium.common.exceptions import TimeoutException
from crazydriver import CrazyDriver, By

In [ ]:
KEYWORD = "lash extension"

SAVE_DIR = Path.cwd() / KEYWORD.replace(" ", "_")
if not SAVE_DIR.exists():
    SAVE_DIR.mkdir(parents=True)

SEARCH_ASINS_PATH = SAVE_DIR / "search_asins.json"
SIBLING_ASINS_PATH = SAVE_DIR / "sibling_asins.json"
PRODUCT_INFO_PATH = SAVE_DIR / "product_info.json"

### STEP 1: 分页拉取搜索结果，收集所有 ASIN

In [ ]:
driver = CrazyDriver()

In [ ]:
def iter_search_asins_by_page(keyword: str):
    """在 Selenium 控制的浏览器中按页抓取亚马逊搜索结果的 ASIN。

    本函数为**生成器**：每执行一次 ``next()``（或 ``for`` 循环前进一步），
    会使用全局 ``driver`` 打开对应页码的搜索 URL，即完成一次**搜索翻页**；
    随后从当前页解析 ``data-asin``，将**本页**得到的 ASIN 列表 ``yield``
    给调用方。

    当某一页无法解析出任何 ASIN 时，生成器结束（不再继续翻页）。

    Args:
        keyword: 搜索关键词，用于构造 ``https://www.amazon.com/s?k=...&page=...``。

    Yields:
        当前页提取到的 ASIN 字符串列表；无 ASIN 时不再产出并结束。
    """
    global driver
    page = 1
    while True:
        url = f"https://www.amazon.com/s?k={keyword}&page={page}"
        driver.get(url)  # 访问搜索页面
        items = driver.explicit_waits(By.XPATH, "//div[@role='listitem']")
        asins = [item.get_attribute('data-asin')
                 for item in items if item.get_attribute('data-asin')]
        # 若本页解析不到 ASIN，视为已无后续商品页，结束生成器。
        if not asins:
            return
        yield asins
        page += 1

In [ ]:
all_asins = []
for asins in iter_search_asins_by_page(KEYWORD):
    all_asins.extend(asins)

all_asins = list(set(all_asins))

with open(SEARCH_ASINS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_asins, f, ensure_ascii=False, indent=4)

### STEP 2：遍历搜索结果中的 ASIN，在各自商品详情页提取兄弟 ASIN

In [ ]:
driver = CrazyDriver()

In [ ]:
def get_sibling_asins(asin: str) -> dict[str, list[str]]:
    """打开商品详情页，从页面源码里解析变体维度数据。"""
    global driver
    url = f"https://amazon.com/dp/{asin}"
    driver.get(url)
    pattern = re.compile(r'"dimensionValuesDisplayData"\s*:\s*({.*?})')
    match_ = pattern.search(driver.page_source)
    if match_:
        return json.loads(match_.group(1))
    return {}

In [ ]:
assert SEARCH_ASINS_PATH.exists(), "search_asins.json 不存在"

with open(SEARCH_ASINS_PATH, "r", encoding="utf-8") as f:
    search_asins = json.load(f)

try:
    with open(SIBLING_ASINS_PATH, encoding="utf-8") as f:
        sibling_asins = json.load(f)
except FileNotFoundError:
    sibling_asins = {}

for asin in search_asins:
    if asin in sibling_asins:
        print(f"exists: {asin} -> {len(sibling_asins[asin])}")
        continue

    result = get_sibling_asins(asin)
    sibling_asins[asin] = result

    with open(SIBLING_ASINS_PATH, "w", encoding="utf-8") as f:
        json.dump(sibling_asins, f, ensure_ascii=False, indent=4)

    print(f"new: {asin} -> {len(result)}")

In [ ]:
# 统计兄弟 ASIN 数量

with open(SIBLING_ASINS_PATH, "r", encoding="utf-8") as f:
    sibling_asins = json.load(f)
total = 0
for asin, siblings in sibling_asins.items():
    if not siblings:
        total += 1
    else:
        total += len(siblings)
print(total)

### STEP 3: 获取所有 ASIN 的商品信息

In [ ]:
driver = CrazyDriver()

In [ ]:
def get_product_info(asin: str) -> dict:
    """打开商品详情页，用 lxml 从 ``page_source`` 解析标题与主价文案。"""
    global driver
    url = f"https://www.amazon.com/dp/{asin}"
    driver.get(url)
    try:
        driver.explicit_wait(By.XPATH, "//*[@id='productTitle']", seconds=10)
        driver.explicit_wait(
            By.XPATH, "//span[@id='apex-pricetopay-accessibility-label']", seconds=10)
    except TimeoutException:
        # 节点未在时限内出现仍解析已加载 HTML，部分页面仍可抠出标题或价。
        pass
    root = etree.HTML(driver.page_source)
    t = root.xpath("//*[@id='productTitle']")
    p = root.xpath(
        "//*[@id='apex-pricetopay-accessibility-label']")
    title = (t[0].text or "").strip() if t else ""
    price = (p[0].text or "").strip() if p else ""

    best_sellers_rank = ""
    for li in root.xpath("//*[@id='detailBullets_feature_div']/ul/li"):
        if not any(s.strip() == "Best Sellers Rank:" for s in li.xpath(".//text()")):
            continue
        best_sellers_rank = " ".join(
            [i.strip() for i in li.xpath(".//text()")[2:] if i.strip()])
        break
    return {
        "asin": asin,
        "url": url,
        "title": title,
        "price": price,
        "best_sellers_rank": best_sellers_rank,
    }

In [ ]:
assert SIBLING_ASINS_PATH.exists(), "sibling_asins.json 不存在"

with open(SIBLING_ASINS_PATH, "r", encoding="utf-8") as f:
    all_sibling_asins = json.load(f)

try:
    with open(PRODUCT_INFO_PATH, "r", encoding="utf-8") as f:
        product_info = defaultdict(list, json.load(f))
except FileNotFoundError:
    product_info = defaultdict(list)

# sibling_asins 包含 original_asin。
original_asin: str
raw_sibling_asins: dict[str, list[str]]
for original_asin, raw_sibling_asins in all_sibling_asins.items():
    sibling_asins = raw_sibling_asins.keys() if raw_sibling_asins else [
        original_asin]

    have_asins = {
        info["asin"] for info in product_info[original_asin] if info is not None
    }
    for asin in sibling_asins:
        if asin in have_asins:
            print(f"exists: {original_asin} -> {asin}")
            continue
        this_product_info = get_product_info(asin)
        this_product_info["tags"] = raw_sibling_asins.get(asin, [])
        product_info[original_asin].append(this_product_info)
        with open(PRODUCT_INFO_PATH, "w", encoding="utf-8") as f:
            json.dump(product_info, f, ensure_ascii=False, indent=4)
        print(f"{asin} -> {this_product_info}")